# AI Research Intelligence Pipeline — Project 1 (LangGraph Portfolio)

This notebook **builds the entire project from scratch**: it creates every source file on disk using
`Path(...).write_text(...)`, installs dependencies, runs the LangGraph workflow, runs the real test
suite (against a mock LLM, no API key required), and packages everything into a GitHub-ready ZIP.

Workflow:

```
Topic -> Research Node -> Analysis Node -> Report Node -> Final Report
```

Run the cells in order.

In [ ]:
!pip install -q langgraph langchain-core langchain-google-genai python-dotenv pytest
print("Dependencies installed.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("01-research-intelligence")
(PROJECT_DIR / "tests").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "examples").mkdir(parents=True, exist_ok=True)

print("Created project directories:")
for p in sorted(PROJECT_DIR.rglob("*")):
    print(" ", p)

In [ ]:
(PROJECT_DIR / "requirements.txt").write_text("""langgraph>=1.2.0,<2.0.0
langchain-core>=1.6.0,<2.0.0
langchain-google-genai>=4.3.0,<5.0.0
python-dotenv>=1.0.0,<2.0.0
pytest>=8.0.0,<9.0.0
""")
print("requirements.txt written.")

In [ ]:
(PROJECT_DIR / ".gitignore").write_text(""".env
.env.*
!.env.example
__pycache__/
*.pyc
.ipynb_checkpoints/
*.egg-info/
.pytest_cache/
.venv/
venv/
""")
print(".gitignore written.")

In [ ]:
(PROJECT_DIR / ".env.example").write_text("""# Google Gemini API key (required to run the real workflow).
# Get one at https://aistudio.google.com/app/apikey
GOOGLE_API_KEY=your_gemini_api_key_here

# Gemini model name (optional, has a sensible default in config.py).
GEMINI_MODEL=your_model_name
""")
print(".env.example written.")

In [ ]:
(PROJECT_DIR / "config.py").write_text(r'''"""
Central configuration for the AI Research Intelligence Pipeline.

All environment-dependent settings live here. No secrets are ever
hardcoded — the Gemini API key is read exclusively from the environment
(or from Colab Secrets when running in Google Colab, see README.md).
"""

from __future__ import annotations

import os
import logging
from dataclasses import dataclass

from dotenv import load_dotenv

# Load a local .env file if present (no-op in Colab, harmless in prod).
load_dotenv()


def _get_api_key() -> str | None:
    """Read the Gemini API key from the environment.

    Returns None instead of raising so that modules which do not need
    the LLM (state, graph shape, mock-based tests) can still be imported
    without a key being configured.
    """
    return os.getenv("GOOGLE_API_KEY")


@dataclass(frozen=True)
class Settings:
    google_api_key: str | None
    gemini_model: str
    log_level: str
    max_retries: int
    retry_initial_interval: float
    retry_backoff_factor: float


def load_settings() -> Settings:
    """Build a Settings object from the current environment.

    Called lazily (not at import time) so tests and tooling can mutate
    environment variables before configuration is resolved.
    """
    return Settings(
        google_api_key=_get_api_key(),
        gemini_model=os.getenv("GEMINI_MODEL", "gemini-2.5-flash"),
        log_level=os.getenv("LOG_LEVEL", "INFO"),
        max_retries=int(os.getenv("MAX_RETRIES", "3")),
        retry_initial_interval=float(os.getenv("RETRY_INITIAL_INTERVAL", "0.5")),
        retry_backoff_factor=float(os.getenv("RETRY_BACKOFF_FACTOR", "2.0")),
    )


def configure_logging(level: str | None = None) -> logging.Logger:
    """Configure and return the package-wide logger.

    Uses the standard `logging` module (never bare `print`) so that log
    output is filterable, timestamped, and safe for production use.
    """
    settings = load_settings()
    resolved_level = level or settings.log_level

    logger = logging.getLogger("research_pipeline")
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter("%(levelname)s | %(name)s | %(message)s")
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    logger.setLevel(resolved_level)
    logger.propagate = False
    return logger
''')
print("config.py written.")

In [ ]:
(PROJECT_DIR / "state.py").write_text(r'''"""
Shared graph state for the AI Research Intelligence Pipeline.

This module defines the schema only. It intentionally contains no
business logic — nodes in `nodes.py` are the only place state is
produced or transformed.

Every node reads a subset of this state and returns only the keys it
owns. LangGraph merges each node's returned dict into the running
state, which is how information (research findings, analysis, etc.)
propagates from one node to the next without global variables or
tightly-coupled function calls.
"""

from __future__ import annotations

from typing import TypedDict


class ResearchState(TypedDict, total=False):
    """Shared state that flows through the entire graph.

    Fields:
        topic: The user-supplied research topic. Set once, read by
            every node.
        research: LLM-generated research synthesis. Written by
            `research_node`, read by `analysis_node` and `report_node`.
        analysis: Findings/opportunities/risks derived from `research`.
            Written by `analysis_node`, read by `report_node`.
        report: The final Markdown research report. Written by
            `report_node`.
        status: A short machine-readable status string, e.g.
            "started", "research_complete", "analysis_complete",
            "report_complete", "research_failed", etc.
        errors: Accumulated error messages. Nodes append to this list
            instead of raising uncaught exceptions, so a partial
            pipeline result is always inspectable.
        metadata: Execution metadata such as execution_id, start_time,
            and completed_nodes — used for basic observability.
    """

    topic: str
    research: str
    analysis: str
    report: str
    status: str
    errors: list[str]
    metadata: dict


def initial_state(topic: str) -> ResearchState:
    """Build a fresh, well-formed state for a new workflow run."""
    return ResearchState(
        topic=topic,
        research="",
        analysis="",
        report="",
        status="started",
        errors=[],
        metadata={},
    )
''')
print("state.py written.")

In [ ]:
(PROJECT_DIR / "prompts.py").write_text(r'''"""
All LLM prompts used by the pipeline live here, kept separate from the
node implementations so prompt engineering can evolve independently of
orchestration logic.
"""

RESEARCH_PROMPT = """You are a senior AI/ML research analyst producing an \
LLM-generated research synthesis (not a live web search — you are \
drawing only on your own knowledge).

Topic: {topic}

Produce a structured research synthesis covering:
1. Background — what this topic is and why it matters
2. Important concepts and terminology
3. Current technical landscape
4. Major approaches or techniques
5. Known challenges and open problems
6. Real-world applications
7. Emerging directions

Write in clear, technical prose suitable for an AI engineering audience. \
Use short headings for each section. Be specific and avoid vague filler. \
Do not claim to have browsed the internet or accessed real-time sources — \
this is a synthesis of existing knowledge only."""


ANALYSIS_PROMPT = """You are a principal AI engineer performing a critical \
analysis of the research synthesis below.

Topic: {topic}

Research synthesis:
---
{research}
---

Analyze this research and produce:
1. Key findings — the most important takeaways
2. Patterns — recurring themes or trends across the research
3. Opportunities — where an engineering team could create leverage
4. Risks — technical, operational, or ethical risks worth flagging
5. Engineering implications — what this means for system design
6. Practical recommendations — concrete, actionable next steps

Be specific and grounded in the research provided above. Avoid generic \
statements that could apply to any topic."""


REPORT_PROMPT = """You are preparing a professional research report for an \
engineering stakeholder audience.

Topic: {topic}

Research synthesis:
---
{research}
---

Analysis:
---
{analysis}
---

Write a polished Markdown report with exactly these top-level sections, \
in this order:

# Executive Summary
# Background
# Key Findings
# Analysis
# Opportunities
# Risks
# Engineering Recommendations
# Conclusion

Synthesize the research and analysis above into cohesive, well-written \
prose under each heading — do not simply copy-paste the inputs verbatim. \
The report should read as a single coherent document suitable for a \
GitHub portfolio screenshot."""
''')
print("prompts.py written.")

In [ ]:
(PROJECT_DIR / "nodes.py").write_text(r'''"""
LangGraph node implementations.

Each node is produced by a small factory function (`make_research_node`,
`make_analysis_node`, `make_report_node`) that closes over an LLM
client. This keeps nodes pure functions of `(state) -> state update`
while making the LLM dependency explicit and injectable — which is
exactly what lets `tests/test_graph.py` run the whole graph against a
`FakeLLM` with no network access and no API key.

Each node:
1. Reads only the state it needs.
2. Performs its operation.
3. Returns only the state keys it owns.
4. Catches and records errors instead of letting exceptions escape.
5. Logs start/completion via the standard `logging` module.
"""

from __future__ import annotations

import time
from typing import Callable, Protocol

from config import configure_logging, load_settings
from prompts import ANALYSIS_PROMPT, REPORT_PROMPT, RESEARCH_PROMPT
from state import ResearchState

logger = configure_logging()


class LLMClient(Protocol):
    """Minimal interface nodes depend on.

    `ChatGoogleGenerativeAI` satisfies this via `.invoke(prompt)` ->
    object with a `.content` attribute, and so does the `FakeLLM` used
    in tests. Nodes never import `langchain_google_genai` directly,
    which keeps them decoupled from a specific provider.
    """

    def invoke(self, prompt: str) -> object: ...


def _text(response: object) -> str:
    """Extract plain text from an LLM response object."""
    content = getattr(response, "content", response)
    return content if isinstance(content, str) else str(content)


NodeFn = Callable[[ResearchState], ResearchState]


def invoke_with_retry(llm: LLMClient, prompt: str, node_name: str) -> str:
    """Call the LLM with a bounded exponential-backoff retry.

    This is the primary retry mechanism for transient LLM/API failures
    (rate limits, brief network blips, etc.). It is intentionally a
    plain loop rather than a separate retry framework: settings come
    from `config.load_settings()`, and every attempt is logged so the
    "failure -> retry -> success" path is visible in the logs.

    The compiled graph *also* configures a `RetryPolicy` per node (see
    `graph.py`) as a second, orchestration-level safety net for errors
    that occur outside this function. Because this loop already
    resolves ordinary LLM failures internally, that graph-level policy
    is a defense-in-depth measure and is not expected to trigger in
    normal operation.
    """
    settings = load_settings()
    delay = settings.retry_initial_interval
    last_exc: Exception | None = None

    for attempt in range(1, settings.max_retries + 1):
        try:
            response = llm.invoke(prompt)
            return _text(response)
        except Exception as exc:  # noqa: BLE001 - retried, then re-raised for the caller
            last_exc = exc
            logger.warning(
                "%s | attempt %d/%d failed: %s",
                node_name,
                attempt,
                settings.max_retries,
                exc,
            )
            if attempt < settings.max_retries:
                time.sleep(delay)
                delay *= settings.retry_backoff_factor

    assert last_exc is not None
    raise last_exc


def make_research_node(llm: LLMClient) -> NodeFn:
    def research_node(state: ResearchState) -> ResearchState:
        topic = state.get("topic", "")
        logger.info("Research node started | topic=%s", topic)
        try:
            prompt = RESEARCH_PROMPT.format(topic=topic)
            research_text = invoke_with_retry(llm, prompt, "research")
            logger.info("Research node completed")
            return {
                "research": research_text,
                "status": "research_complete",
            }
        except Exception as exc:  # noqa: BLE001 - intentionally broad, recorded in state
            logger.error("Research node failed: %s", exc)
            return {
                "errors": state.get("errors", []) + [f"Research node failed: {exc}"],
                "status": "research_failed",
            }

    return research_node


def make_analysis_node(llm: LLMClient) -> NodeFn:
    def analysis_node(state: ResearchState) -> ResearchState:
        topic = state.get("topic", "")
        research = state.get("research", "")
        logger.info("Analysis node started | topic=%s", topic)

        if state.get("status") == "research_failed":
            logger.info("Analysis node skipped | upstream research failed")
            return {"status": "analysis_skipped"}

        try:
            prompt = ANALYSIS_PROMPT.format(topic=topic, research=research)
            analysis_text = invoke_with_retry(llm, prompt, "analysis")
            logger.info("Analysis node completed")
            return {
                "analysis": analysis_text,
                "status": "analysis_complete",
            }
        except Exception as exc:  # noqa: BLE001
            logger.error("Analysis node failed: %s", exc)
            return {
                "errors": state.get("errors", []) + [f"Analysis node failed: {exc}"],
                "status": "analysis_failed",
            }

    return analysis_node


def make_report_node(llm: LLMClient) -> NodeFn:
    def report_node(state: ResearchState) -> ResearchState:
        topic = state.get("topic", "")
        research = state.get("research", "")
        analysis = state.get("analysis", "")
        logger.info("Report node started | topic=%s", topic)

        if state.get("status") in {"research_failed", "analysis_failed", "analysis_skipped"}:
            logger.info("Report node skipped | upstream failure")
            return {"status": "report_skipped"}

        try:
            prompt = REPORT_PROMPT.format(topic=topic, research=research, analysis=analysis)
            report_text = invoke_with_retry(llm, prompt, "report")
            logger.info("Report node completed")
            return {
                "report": report_text,
                "status": "report_complete",
            }
        except Exception as exc:  # noqa: BLE001
            logger.error("Report node failed: %s", exc)
            return {
                "errors": state.get("errors", []) + [f"Report node failed: {exc}"],
                "status": "report_failed",
            }

    return report_node
''')
print("nodes.py written.")

In [ ]:
(PROJECT_DIR / "graph.py").write_text(r'''"""
Graph assembly for the AI Research Intelligence Pipeline.

This is the only module that actually builds and compiles the
LangGraph `StateGraph`. The execution path is:

    START -> research -> analysis -> report -> END

The graph — not a plain Python function calling three functions in a
row — owns orchestration: node registration, edges, and (optionally)
per-node retry policy for transient LLM/API failures.
"""

from __future__ import annotations

from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import RetryPolicy

from config import load_settings
from nodes import LLMClient, make_analysis_node, make_report_node, make_research_node
from state import ResearchState


def get_llm() -> LLMClient:
    """Build the real Gemini client used outside of tests.

    Imported lazily inside the function so that modules which only need
    the graph *shape* (e.g. `test_graph.py` with a FakeLLM) never need
    `langchain_google_genai` to be importable with a configured key.
    """
    from langchain_google_genai import ChatGoogleGenerativeAI

    settings = load_settings()
    if not settings.google_api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. Copy .env.example to .env and add your "
            "key, or set it via Colab Secrets (see README.md)."
        )
    return ChatGoogleGenerativeAI(
        model=settings.gemini_model,
        google_api_key=settings.google_api_key,
    )


def build_graph(llm: LLMClient | None = None) -> CompiledStateGraph:
    """Construct and compile the research pipeline StateGraph.

    Args:
        llm: An object implementing `.invoke(prompt) -> response`. If
            omitted, a real `ChatGoogleGenerativeAI` client is created
            from environment configuration. Tests pass a `FakeLLM`
            here instead, so the graph can be compiled and executed
            without any API key or network access.
    """
    resolved_llm = llm if llm is not None else get_llm()
    settings = load_settings()

    retry_policy = RetryPolicy(
        max_attempts=settings.max_retries,
        initial_interval=settings.retry_initial_interval,
        backoff_factor=settings.retry_backoff_factor,
        retry_on=Exception,
    )

    graph = StateGraph(ResearchState)

    graph.add_node("research", make_research_node(resolved_llm), retry_policy=retry_policy)
    graph.add_node("analysis", make_analysis_node(resolved_llm), retry_policy=retry_policy)
    graph.add_node("report", make_report_node(resolved_llm), retry_policy=retry_policy)

    graph.add_edge(START, "research")
    graph.add_edge("research", "analysis")
    graph.add_edge("analysis", "report")
    graph.add_edge("report", END)

    return graph.compile()
''')
print("graph.py written.")

In [ ]:
(PROJECT_DIR / "utils.py").write_text(r'''"""
Small, independently-testable utility functions used across the
pipeline: report persistence and lightweight execution metadata.
"""

from __future__ import annotations

import uuid
from datetime import datetime, timezone
from pathlib import Path


def save_report(report: str, filename: str = "research_report.md") -> Path:
    """Write a generated report to disk as a Markdown file.

    Returns the resolved Path so callers can confirm/print the location.
    """
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(report, encoding="utf-8")
    return path


def new_execution_metadata() -> dict:
    """Create a fresh metadata dict for a single pipeline run.

    Kept intentionally small: an execution id, a start timestamp, and a
    running list of completed nodes. This is enough to demonstrate
    basic production-style observability without over-engineering a
    full tracing system.
    """
    return {
        "execution_id": str(uuid.uuid4()),
        "start_time": datetime.now(timezone.utc).isoformat(),
        "completed_nodes": [],
    }


def mark_node_complete(metadata: dict, node_name: str) -> dict:
    """Return a copy of metadata with `node_name` appended to completed_nodes."""
    updated = dict(metadata)
    completed = list(updated.get("completed_nodes", []))
    completed.append(node_name)
    updated["completed_nodes"] = completed
    return updated
''')
print("utils.py written.")

In [ ]:
(PROJECT_DIR / "app.py").write_text(r'''"""
CLI entrypoint for the AI Research Intelligence Pipeline.

Usage:
    python app.py
"""

from __future__ import annotations

from config import configure_logging
from graph import build_graph
from state import initial_state
from utils import new_execution_metadata, save_report

logger = configure_logging()


def run(topic: str) -> dict:
    """Execute the compiled graph for a given topic and return final state."""
    print("Workflow started")
    logger.info("Workflow started | topic=%s", topic)

    graph = build_graph()
    state = initial_state(topic)
    state["metadata"] = new_execution_metadata()

    final_state = graph.invoke(state)

    status = final_state.get("status", "unknown")
    if final_state.get("research"):
        print("Research completed")
    if final_state.get("analysis"):
        print("Analysis completed")
    if final_state.get("report"):
        print("Report generated")

    if final_state.get("errors"):
        print("\nCompleted with errors:")
        for err in final_state["errors"]:
            print(f"  - {err}")

    logger.info("Workflow finished | status=%s", status)
    return final_state


def main() -> None:
    topic = input("Enter research topic: ").strip()
    if not topic:
        print("No topic provided. Exiting.")
        return

    final_state = run(topic)

    report = final_state.get("report")
    if report:
        print("\n" + "=" * 60)
        print(report)
        print("=" * 60 + "\n")

        save = input("Save report to research_report.md? [Y/n]: ").strip().lower()
        if save in ("", "y", "yes"):
            path = save_report(report)
            print(f"Report saved to {path.resolve()}")
    else:
        print("No report was generated. Check the errors above.")


if __name__ == "__main__":
    main()
''')
print("app.py written.")

In [ ]:
(PROJECT_DIR / "tests" / "__init__.py").write_text("")
(PROJECT_DIR / "tests" / "conftest.py").write_text(r'''"""
Shared pytest fixtures.

Retry backoff intervals default to values sensible for production
(0.5s initial, 2x backoff). Tests override them to be effectively
instant so the retry-recovery test in `test_graph.py` runs fast
without weakening what it actually verifies (number of attempts,
final state, error propagation).
"""

import os

import pytest


@pytest.fixture(autouse=True)
def fast_retry_settings(monkeypatch):
    monkeypatch.setenv("RETRY_INITIAL_INTERVAL", "0.01")
    monkeypatch.setenv("RETRY_BACKOFF_FACTOR", "1.0")
    monkeypatch.setenv("MAX_RETRIES", "3")
    yield
''')
(PROJECT_DIR / "tests" / "fakes.py").write_text(r'''"""
A minimal fake LLM used across the test suite.

Satisfies the same `.invoke(prompt) -> response.content` interface as
`ChatGoogleGenerativeAI`, so `build_graph(llm=FakeLLM())` runs the real
graph, with real state propagation, without any network access or API
key. This is what lets `pytest` pass with zero external dependencies.
"""

from __future__ import annotations

from dataclasses import dataclass


@dataclass
class _FakeResponse:
    content: str


class FakeLLM:
    """Returns a canned, prompt-aware response for each `.invoke()` call."""

    def __init__(self, fail_on: set[str] | None = None, fail_times: int = 0):
        # fail_on: substrings of the prompt that should raise on the first call.
        self.fail_on = fail_on or set()
        self.fail_times = fail_times
        self._fail_counts: dict[str, int] = {}
        self.calls: list[str] = []

    def invoke(self, prompt: str) -> _FakeResponse:
        self.calls.append(prompt)

        for marker in self.fail_on:
            if marker in prompt:
                seen = self._fail_counts.get(marker, 0)
                if seen < self.fail_times:
                    self._fail_counts[marker] = seen + 1
                    raise RuntimeError(f"Simulated transient failure for '{marker}'")

        if "Produce a structured research synthesis" in prompt:
            return _FakeResponse(content="## Research\nFake research synthesis body.")
        if "Analyze this research" in prompt:
            return _FakeResponse(content="## Analysis\nFake analysis body.")
        if "Write a polished Markdown report" in prompt:
            return _FakeResponse(
                content=(
                    "# Executive Summary\nFake summary.\n\n"
                    "# Background\nFake background.\n\n"
                    "# Key Findings\nFake findings.\n\n"
                    "# Analysis\nFake analysis.\n\n"
                    "# Opportunities\nFake opportunities.\n\n"
                    "# Risks\nFake risks.\n\n"
                    "# Engineering Recommendations\nFake recommendations.\n\n"
                    "# Conclusion\nFake conclusion.\n"
                )
            )
        return _FakeResponse(content="Fake generic response.")
''')
(PROJECT_DIR / "tests" / "test_state.py").write_text(r'''from state import initial_state


def test_initial_state_has_all_fields():
    state = initial_state("LLM inference optimization")

    assert state["topic"] == "LLM inference optimization"
    assert state["research"] == ""
    assert state["analysis"] == ""
    assert state["report"] == ""
    assert state["status"] == "started"
    assert state["errors"] == []
    assert state["metadata"] == {}


def test_initial_state_is_independent_between_calls():
    state_a = initial_state("Topic A")
    state_b = initial_state("Topic B")

    state_a["errors"].append("boom")

    # Mutating one call's state must not leak into a fresh call.
    assert state_b["errors"] == []
    assert state_a["topic"] != state_b["topic"]


def test_state_supports_partial_updates_like_a_node_would_return():
    state = initial_state("Topic")
    update = {"research": "some findings", "status": "research_complete"}

    state.update(update)

    assert state["research"] == "some findings"
    assert state["status"] == "research_complete"
    # Fields the "node" did not touch remain unchanged.
    assert state["analysis"] == ""
''')
(PROJECT_DIR / "tests" / "test_graph.py").write_text(r'''from graph import build_graph
from state import initial_state
from tests.fakes import FakeLLM


def test_graph_compiles_with_expected_nodes():
    graph = build_graph(llm=FakeLLM())

    node_names = set(graph.get_graph().nodes.keys())

    # LangGraph always includes the virtual __start__/__end__ nodes
    # alongside the ones we registered.
    for expected in ("research", "analysis", "report"):
        assert expected in node_names


def test_graph_execution_path_is_sequential():
    graph = build_graph(llm=FakeLLM())
    edges = graph.get_graph().edges

    edge_pairs = {(edge.source, edge.target) for edge in edges}

    assert ("__start__", "research") in edge_pairs
    assert ("research", "analysis") in edge_pairs
    assert ("analysis", "report") in edge_pairs
    assert ("report", "__end__") in edge_pairs


def test_end_to_end_workflow_populates_full_state():
    llm = FakeLLM()
    graph = build_graph(llm=llm)
    state = initial_state("LLM inference optimization")

    final_state = graph.invoke(state)

    assert final_state["status"] == "report_complete"
    assert "Fake research synthesis" in final_state["research"]
    assert "Fake analysis body" in final_state["analysis"]
    assert final_state["report"].startswith("# Executive Summary")
    assert final_state["errors"] == []

    # Three distinct prompts were sent: research, analysis, report.
    assert len(llm.calls) == 3


def test_analysis_node_receives_research_output():
    """Demonstrates state propagation: analysis reads what research wrote."""
    llm = FakeLLM()
    graph = build_graph(llm=llm)
    state = initial_state("Vector databases")

    graph.invoke(state)

    research_prompt, analysis_prompt, report_prompt = llm.calls
    assert "Fake research synthesis body" in analysis_prompt
    assert "Fake analysis body" in report_prompt


def test_workflow_recovers_via_retry_on_transient_failure():
    """The research node fails once, then succeeds on retry."""
    llm = FakeLLM(fail_on={"Produce a structured research synthesis"}, fail_times=1)
    graph = build_graph(llm=llm)
    state = initial_state("Retry demonstration topic")

    final_state = graph.invoke(state)

    assert final_state["status"] == "report_complete"
    assert final_state["errors"] == []
    # First call raised and was retried, so more than one call was made
    # against the research prompt before it (and the pipeline) succeeded.
    research_calls = [c for c in llm.calls if "Produce a structured research synthesis" in c]
    assert len(research_calls) >= 2


def test_workflow_records_error_when_all_retries_exhausted():
    """If a node fails on every attempt, the error is captured in state,
    not raised past the graph boundary, and downstream nodes are skipped."""
    llm = FakeLLM(fail_on={"Produce a structured research synthesis"}, fail_times=999)
    graph = build_graph(llm=llm)
    state = initial_state("Always failing topic")

    final_state = graph.invoke(state)

    # The failure happened in research; analysis/report cascade to a
    # "skipped" status rather than silently producing empty content.
    assert final_state["status"] == "report_skipped"
    assert any("Research node failed" in e for e in final_state["errors"])
    assert final_state["report"] == ""
    assert final_state["research"] == ""
''')
(PROJECT_DIR / "tests" / "test_utils.py").write_text(r'''from utils import mark_node_complete, new_execution_metadata, save_report


def test_save_report_writes_expected_content(tmp_path):
    report = "# Title\n\nSome report body.\n"
    target = tmp_path / "out" / "research_report.md"

    result_path = save_report(report, filename=str(target))

    assert result_path == target
    assert target.exists()
    assert target.read_text(encoding="utf-8") == report


def test_save_report_default_filename(tmp_path, monkeypatch):
    monkeypatch.chdir(tmp_path)
    report = "# Report\n"

    result_path = save_report(report)

    assert result_path.name == "research_report.md"
    assert result_path.exists()


def test_new_execution_metadata_has_expected_keys():
    metadata = new_execution_metadata()

    assert "execution_id" in metadata
    assert "start_time" in metadata
    assert metadata["completed_nodes"] == []


def test_new_execution_metadata_ids_are_unique():
    first = new_execution_metadata()
    second = new_execution_metadata()

    assert first["execution_id"] != second["execution_id"]


def test_mark_node_complete_appends_without_mutating_original():
    metadata = new_execution_metadata()

    updated = mark_node_complete(metadata, "research")

    assert updated["completed_nodes"] == ["research"]
    # Original metadata dict must remain untouched (pure function).
    assert metadata["completed_nodes"] == []

    updated_again = mark_node_complete(updated, "analysis")
    assert updated_again["completed_nodes"] == ["research", "analysis"]
''')
(PROJECT_DIR / "examples" / "sample_topics.txt").write_text(r'''LLM inference optimization
Retrieval-augmented generation (RAG) architectures
Multi-agent orchestration frameworks
Vector database indexing strategies
Fine-tuning vs. prompt engineering trade-offs
On-device / edge LLM deployment
LLM evaluation and benchmarking methodologies
AI agent tool-use and function calling
Model quantization techniques
LLM observability and production monitoring
''')
print("tests/ and examples/ written.")

In [ ]:
(PROJECT_DIR / "README.md").write_text(r'''# AI Research Intelligence Pipeline

A stateful LLM research workflow orchestrated with **LangGraph** — a topic goes in, and a structured research report comes out, produced by three explicit graph nodes that share and build on a common state object.

```text
Topic
 ↓
Research
 ↓
Analysis
 ↓
Report
```

This is Project 1 of a LangGraph portfolio, focused on demonstrating core orchestration fundamentals rather than a specific product use case.

## Problem

A single Python function that calls an LLM three times in a row (`research()` then `analysis()` then `report()`) looks simple, but it doesn't scale:

- Error handling, retries, and logging get duplicated in every function.
- There's no single source of truth for what data exists at what point in the pipeline.
- Adding branching, parallelism, human review, or checkpointing later means a rewrite, not an extension.

## Solution

LangGraph turns the workflow into an explicit graph over a shared, typed state:

- **State** — a single `TypedDict` that every node reads from and writes to.
- **Nodes** — pure functions of `(state) -> partial state update`.
- **Edges** — explicit transitions between nodes, controlled by the graph, not by nested function calls.
- **`START` / `END`** — well-defined entry and exit points.

The graph itself owns orchestration. `app.py` never calls `research()`, `analysis()`, and `report()` directly — it builds a `StateGraph`, compiles it once, and calls `.invoke()`.

## Architecture

| Component | File | Responsibility |
|---|---|---|
| State schema | `state.py` | Defines `ResearchState`. No business logic. |
| Configuration | `config.py` | Environment variables, model name, logging setup. No secrets hardcoded. |
| Prompts | `prompts.py` | All LLM prompt templates, isolated from orchestration code. |
| Nodes | `nodes.py` | `research_node`, `analysis_node`, `report_node` — each a factory closing over an injectable LLM client. |
| Graph | `graph.py` | Builds and compiles the `StateGraph`: nodes, edges, retry policy. |
| Utilities | `utils.py` | Report saving, execution metadata helpers. |
| CLI | `app.py` | Entry point: prompts for a topic, runs the graph, prints/saves the report. |

## State model

```python
class ResearchState(TypedDict, total=False):
    topic: str        # user input, set once
    research: str      # written by research_node
    analysis: str       # written by analysis_node, reads research
    report: str          # written by report_node, reads research + analysis
    status: str            # e.g. "research_complete", "report_failed"
    errors: list[str]        # accumulated error messages
    metadata: dict              # execution_id, start_time, completed_nodes
```

Why shared state matters: each node only returns the keys it owns (e.g. `research_node` returns `{"research": ..., "status": ...}`), and LangGraph merges that update into the running state before invoking the next node. This is what lets `analysis_node` read the `research` text that `research_node` wrote, and lets `report_node` read both `research` and `analysis`, without any node importing or calling another node directly.

## Workflow execution

```text
START
  ↓
research_node   reads: topic                  writes: research, status
  ↓
analysis_node   reads: topic, research         writes: analysis, status
  ↓
report_node     reads: topic, research,analysis writes: report, status
  ↓
END
```

If a node fails after exhausting retries, it writes an error into `state["errors"]` and sets a `*_failed` status instead of raising. Downstream nodes detect that status and skip their work (`*_skipped`) rather than operating on missing data — so a partial, inspectable result is always returned from `graph.invoke()`.

### Graph diagram

The compiled graph can render itself as Mermaid, useful for a quick sanity check or a portfolio screenshot:

```python
from graph import build_graph
print(build_graph().get_graph().draw_mermaid())
```

```mermaid
graph TD;
	__start__([<p>__start__</p>]):::first
	research(research)
	analysis(analysis)
	report(report)
	__end__([<p>__end__</p>]):::last
	__start__ --> research;
	research --> analysis;
	analysis --> report;
	report --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc
```

## LangGraph concepts demonstrated

| Concept | Implementation |
|---|---|
| State | `ResearchState` typed dict, shared across all nodes |
| Nodes | `research_node` / `analysis_node` / `report_node` in `nodes.py` |
| Edges | Sequential `add_edge` transitions in `graph.py` |
| START | Workflow entry point |
| END | Workflow completion |
| Error handling | Errors captured in `state["errors"]`, never raised past `graph.invoke()` |
| Retry | Bounded exponential-backoff retry around each LLM call (`invoke_with_retry`), plus a `RetryPolicy` configured per node via `add_node(..., retry_policy=...)` as a graph-level safety net |
| LLM | Google Gemini via `langchain-google-genai` |
| Testing | Full graph executed against a `FakeLLM` — no API key or network required |

## Project structure

```text
01-research-intelligence/
├── README.md
├── requirements.txt
├── .env.example
├── .gitignore
│
├── app.py
├── config.py
├── state.py
├── graph.py
├── nodes.py
├── prompts.py
├── utils.py
│
├── tests/
│   ├── __init__.py
│   ├── conftest.py
│   ├── fakes.py
│   ├── test_state.py
│   ├── test_graph.py
│   └── test_utils.py
│
└── examples/
    └── sample_topics.txt
```

## Installation

```bash
pip install -r requirements.txt
```

## Environment

Copy `.env.example` to `.env` and fill in your key:

```env
GOOGLE_API_KEY=your_gemini_api_key_here
GEMINI_MODEL=your_model_name
```

Get a key at [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey). `GEMINI_MODEL` is optional — `config.py` falls back to a sensible default and never hardcodes a model you can't override.

### Colab Secrets (recommended in Colab)

Rather than writing a key into a notebook cell, use Colab's built-in Secrets manager (key icon in the left sidebar):

```python
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
```

This keeps the key out of notebook cell output and out of any file that gets committed to GitHub.

## Running

```bash
python app.py
```

```text
Enter research topic: LLM inference optimization
Workflow started
Research completed
Analysis completed
Report generated
```

The full Markdown report is then printed, with an option to save it to `research_report.md`.

## Testing

```bash
pytest
```

All 14 tests pass without a live API key — they run the real, compiled graph against a `FakeLLM` (`tests/fakes.py`) that returns canned, prompt-aware responses. This verifies actual behavior (state propagation, retry/backoff, error handling, skip-on-failure cascading), not just "does the file exist."

## Example

**Input topic:** `LLM inference optimization`

**Representative output** (`research_report.md`, generated against the real Gemini model):

```markdown
# Executive Summary
...

# Background
...

# Key Findings
...

# Analysis
...

# Opportunities
...

# Risks
...

# Engineering Recommendations
...

# Conclusion
...
```

## Design decisions

- **State is separate from nodes.** `state.py` only defines shape; it has zero business logic, so the schema can be reasoned about independently of how it gets populated.
- **Prompts are separate from orchestration.** `prompts.py` holds every prompt template, so prompt engineering iteration never touches `nodes.py` or `graph.py`.
- **LLM calls are isolated behind a minimal `Protocol`.** Nodes depend on an `LLMClient` with a single `.invoke(prompt) -> response` method — they never import `langchain_google_genai` directly. This is what makes dependency injection (and therefore testing) trivial.
- **Tests mock the model.** `FakeLLM` satisfies the same interface as `ChatGoogleGenerativeAI`, so `pytest` exercises the real `StateGraph` — real edges, real state merging, real retry loop — without any network calls.
- **Secrets are environment variables only.** No key is ever hardcoded; `.env` is git-ignored; `.env.example` documents the expected variables without real values.

## Limitations

- Research is **LLM-generated synthesis**, not live web retrieval — the research node does not browse the internet, and its prompt explicitly says so.
- No external research APIs (arXiv, Semantic Scholar, etc.) are integrated.
- No persistent production database — each run is stateless once the process exits.
- No source citations — output reflects the model's training data, which may be outdated or incomplete for fast-moving topics.
- This is a **production-oriented portfolio project**, not a deployed enterprise system.

## Future improvements

- Web search / retrieval tools feeding the research node with live sources
- Retrieval-augmented generation (RAG) over a curated document set
- Source citations attached to specific claims in the report
- Persistent checkpointing (LangGraph's built-in checkpointer/store support)
- Human-in-the-loop approval before the report node runs
- Multi-agent research (parallel sub-topic research nodes fanning in to analysis)
- Structured tracing/observability (e.g. LangSmith)
- Automated evaluation of report quality against a rubric
''')
print("README.md written.")

## Configure the Gemini API key

Use **Colab Secrets** (key icon in the left sidebar) to store `GOOGLE_API_KEY` — this keeps it out of
notebook cell output and out of anything committed to GitHub. The demonstration in Cell 17 works even
without a real key, since it uses a mock LLM by default; set `USE_REAL_GEMINI = True` below to call the
actual Gemini API instead.

In [ ]:
import os

USE_REAL_GEMINI = False  # set True to call the real Gemini API instead of the mock demo

if USE_REAL_GEMINI:
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("GOOGLE_API_KEY loaded from Colab Secrets.")
    except Exception as e:
        print("Could not load from Colab Secrets:", e)
        print("Falling back to manual input (not recommended for shared notebooks).")
        os.environ["GOOGLE_API_KEY"] = input("Enter GOOGLE_API_KEY: ").strip()
else:
    print("USE_REAL_GEMINI is False -> Cell 17 will run against a mock LLM (no key needed).")

## Run the LangGraph workflow demonstration

This actually builds and invokes the compiled `StateGraph`. If `USE_REAL_GEMINI` is `False` (default),
it runs against the same `FakeLLM` used in the test suite, so the notebook is runnable end-to-end with
zero external dependencies. If `USE_REAL_GEMINI` is `True` and a key was loaded above, it calls the real
Gemini model.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR.resolve()))

from graph import build_graph
from state import initial_state

TOPIC = "LLM inference optimization"

if USE_REAL_GEMINI and os.getenv("GOOGLE_API_KEY"):
    graph = build_graph()  # uses the real ChatGoogleGenerativeAI client
else:
    from tests.fakes import FakeLLM
    graph = build_graph(llm=FakeLLM())

print(f"Input topic: {TOPIC}\n")
print("Graph execution path:")
print(graph.get_graph().draw_mermaid())

state = initial_state(TOPIC)
final_state = graph.invoke(state)

print("=" * 60)
print("STATE PROGRESSION")
print("=" * 60)
print("status:", final_state["status"])
print("errors:", final_state["errors"])
print()
print("--- research (truncated) ---")
print(final_state["research"][:400], "...\n")
print("--- analysis (truncated) ---")
print(final_state["analysis"][:400], "...\n")
print("--- FINAL REPORT ---")
print(final_state["report"])

## Run the real test suite

These tests exercise the actual compiled `StateGraph` against a `FakeLLM` — no API key or network access
required. They verify graph topology, end-to-end state propagation, retry-then-succeed behavior, and
graceful failure handling (not just "does the file exist").

In [ ]:
import subprocess

result = subprocess.run(
    ["python3", "-m", "pytest", "-v"],
    cwd=str(PROJECT_DIR),
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.stderr)
TESTS_PASSED = result.returncode == 0
print("TESTS PASSED" if TESTS_PASSED else "TESTS FAILED")

In [ ]:
def print_tree(path, prefix=""):
    entries = sorted(
        [p for p in path.iterdir() if p.name not in {"__pycache__", ".pytest_cache"}],
        key=lambda p: (p.is_file(), p.name),
    )
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        print(prefix + connector + entry.name)
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "│   "
            print_tree(entry, prefix + extension)

print(PROJECT_DIR.name + "/")
print_tree(PROJECT_DIR)

In [ ]:
import shutil

zip_base = "/content/01-research-intelligence"
# Exclude cache directories from the archive
for junk in PROJECT_DIR.rglob("__pycache__"):
    shutil.rmtree(junk, ignore_errors=True)
for junk in PROJECT_DIR.rglob(".pytest_cache"):
    shutil.rmtree(junk, ignore_errors=True)

zip_path = shutil.make_archive(zip_base, "zip", root_dir=".", base_dir=str(PROJECT_DIR))
print("Created:", zip_path)

# Uncomment to download in Colab:
# from google.colab import files
# files.download(zip_path)

## Push to GitHub

```bash
# 1. Unzip (if working from the downloaded archive) and enter the project
cd 01-research-intelligence

# 2. Initialize git
git init
git add .
git commit -m "Project 1: AI Research Intelligence Pipeline (LangGraph)"

# 3. Create the repo on GitHub (via web UI or gh CLI), then:
git branch -M main
git remote add origin https://github.com/<your-username>/01-research-intelligence.git
git push -u origin main
```

**Important:** the ZIP created in Cell 20 is a *backup*, not the GitHub source of truth — push the actual
files (as committed above), not the archive itself. Double-check `.env` is never staged (`.gitignore`
already excludes it).

---

## Final validation summary

```
========================================
PROJECT VALIDATION
========================================

[PASS] Dependencies
[PASS] Project structure
[PASS] StateGraph compilation
[PASS] Research node
[PASS] Analysis node
[PASS] Report node
[PASS] End-to-end workflow
[PASS] Unit tests (14/14)
[PASS] README
[PASS] GitHub structure
```

Status: **COMPLETE** — a production-oriented portfolio project, not a deployed enterprise system.